# Simulating some NR events

Pueh Leng Tan, 10 March 2026

In [1]:
import os
from time import time

import numpy as np
import pandas as pd
import scipy.stats as sps
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import jax.numpy as jnp
import multihist as mh
import json

import appletree as apt
from appletree.utils import get_file_path

import aptext

XLA_PYTHON_CLIENT_PREALLOCATE is set to false
XLA_PYTHON_CLIENT_ALLOCATOR is set to platform
Using aptext package from https://github.com/XENONnT/applefiles


In [2]:
# constrain the GPU memory usage
apt.set_gpu_memory_usage(0.5)

## Define component

### ComponentSim

In [3]:
# Initialize component
#nr = apt.NR(bins=[bins_cs1, bins_cs2], bins_type="irreg")
#nr = apt.NR()
#nr = apt.components.NRBand2f()
nr = apt.components.AmBe()

# i should write my own stuff (or use minghao's) under components of aptext of apt
# this component tells me exactly which steps to do, just like registering cuts in cutax
# https://github.com/XENONnT/applefiles/blob/cbb3139150526e647d1e2985f2079577f8218cd3/aptext/components/two_fold.py#L121-L146

In [22]:
aa = nr.show_config()
# all the ingredients it needs
# if current = None, means it uses default

In [23]:
aa

,option,default,current,applies_to,help
0,s1_bias_3f,_s1_bias.json,/home/puehlengt/appletree/appletree/maps/_s1_b...,[s1_area],3fold S1 reconstruction bias
1,s1_smear_3f,_s1_smearing.json,/home/puehlengt/appletree/appletree/maps/_s1_s...,[s1_area],3fold S1 reconstruction smearing
2,s1_correction,_s1_correction.json,/home/puehlengt/appletree/appletree/maps/_s1_c...,[s1_correction],S1 xyz correction on reconstructed positions
3,s1_lce,_s1_correction.json,/home/puehlengt/appletree/appletree/maps/_s1_c...,[s1_lce],S1 light collection efficiency
4,g4,"[g4_ambe_sr0.npy, 1000000, 7]","[g4_ambe_sr0.npy, 1000000, 7]","[energy, x, y, z, time, s2_tag, event_id, g4_e...",full-chain geant4 input file
5,literature_field,23.0,23.0,[ThomasImel],Drift field in each literature
6,s2_bias,_s2_bias.json,/home/puehlengt/appletree/appletree/maps/_s2_b...,[s2_area_naive],S2 reconstruction bias
7,s2_smear,_s2_smearing.json,/home/puehlengt/appletree/appletree/maps/_s2_s...,[s2_area_naive],S2 reconstruction smearing
8,s2_bias,_s2_bias.json,/home/puehlengt/appletree/appletree/maps/_s2_b...,[alt_s2_area_naive],S2 reconstruction bias
9,s2_smear,_s2_smearing.json,/home/puehlengt/appletree/appletree/maps/_s2_s...,[alt_s2_area_naive],S2 reconstruction smearing


In [6]:
f_instruct = 'instruct_ambe_realistic.json'
with open(f_instruct, 'rb') as fid:
    instruct = json.load(fid)

In [7]:
my_config = {} # get this from miao when using his component in the future

for _opt in aa['option']:
    if _opt in instruct['configs']:
        my_config[_opt] = instruct['configs'][_opt]

In [21]:
instruct['configs']['g4'], my_config['g4'], aa['option']

(['g4_ambe_sr1.npy', 1000000, 7],
 ['g4_ambe_sr1.npy', 1000000, 7],
 0                                 s1_bias_3f
 1                                s1_smear_3f
 2                              s1_correction
 3                                     s1_lce
 4                                         g4
 5                           literature_field
 6                                    s2_bias
 7                                   s2_smear
 8                                    s2_bias
 9                                   s2_smear
 10                               posrec_reso
 11                         gas_gain_relative
 12                                    s2_lce
 13                                     elife
 14                                       civ
 15                         gas_gain_relative
 16                                    s2_lce
 17                             s2_correction
 18                             s2_correction
 19                          s1_eff_n_hits_3f
 20         

In [8]:
nr.set_config(my_config)
# this must be done before nr.compile

/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_s1_bias_bkg_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_s1_bias_bkg_sr1.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_s1_smearing_bkg_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_s1_smearing_bkg_sr1.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load s1_correction_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/s1_correction_sr1.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load s1_correction_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/s1_correction_sr1.json
  warn(f"Load {fname} successfully from {fpath}")
/hom

In [12]:
# Deduce the workflow(datastructure)
nr.deduce(data_names=["cs1", "cs2"], func_name="simulate")  # 'eff'(efficiency) is always simulated
# above line basically says that i want cs1, cs2, go deduce and gather ingredients required to compute those

nr.rate_name = "nr_rate"  # also we have to specify a normalization factor of the component

# Compile NR script
# This is meta-programing because  appletree can generate codes dynamically
nr.compile()

/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load g4_ambe_sr0.npy successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/g4_ambe_sr0.npy
  warn(f"Load {fname} successfully from {fpath}")


There are 319079 events, 1000476 main/alt S2, which are not equal to batch_size.


/home/puehlengt/applefiles/aptext/multiscatter/g4.py:105: UserWarning: Efficiency is not provided in /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/g4_ambe_sr0.npy, set to 1.
  warn(f"Efficiency is not provided in {self.file_path}, set to 1.")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load civ_sr0.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/civ_sr0.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load _anti_correlation_eff.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/_anti_correlation_eff.json
  warn(f"Load {fname} successfully from {fpath}")


AmBe_llh's map s2_cut_acc is using the parameter s2_cut_acc_sigma.
AmBe_llh's map s1_cut_acc is using the parameter s1_cut_acc_sigma.
AmBe_llh's map s1_eff_n_hits_3f is using the parameter s1_eff_n_hits_3f_sigma.


/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_median_sr0.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_recon_eff_n_hits_median_sr0.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_median_sr0.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_recon_eff_n_hits_median_sr0.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_lower_sr0.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_recon_eff_n_hits_lower_sr0.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_lower_sr0.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../ap

In [ ]:
# apt.clear_cache() # to clear cache, if you want to re-compile the component

In [ ]:
# apt.share._cached_configs, apt.share._cached_functions

In [16]:
nr.show_config()

,option,default,current,applies_to,help
0,s1_bias_3f,_s1_bias.json,/home/puehlengt/appletree/appletree/maps/_s1_b...,[s1_area],3fold S1 reconstruction bias
1,s1_smear_3f,_s1_smearing.json,/home/puehlengt/appletree/appletree/maps/_s1_s...,[s1_area],3fold S1 reconstruction smearing
2,s1_correction,_s1_correction.json,/home/puehlengt/appletree/appletree/maps/_s1_c...,[s1_correction],S1 xyz correction on reconstructed positions
3,s1_lce,_s1_correction.json,/home/puehlengt/appletree/appletree/maps/_s1_c...,[s1_lce],S1 light collection efficiency
4,g4,"[g4_ambe_sr0.npy, 1000000, 7]","[g4_ambe_sr0.npy, 1000000, 7]","[energy, x, y, z, time, s2_tag, event_id, g4_e...",full-chain geant4 input file
5,literature_field,23.0,23.0,[ThomasImel],Drift field in each literature
6,s2_bias,_s2_bias.json,/home/puehlengt/appletree/appletree/maps/_s2_b...,[s2_area_naive],S2 reconstruction bias
7,s2_smear,_s2_smearing.json,/home/puehlengt/appletree/appletree/maps/_s2_s...,[s2_area_naive],S2 reconstruction smearing
8,s2_bias,_s2_bias.json,/home/puehlengt/appletree/appletree/maps/_s2_b...,[alt_s2_area_naive],S2 reconstruction bias
9,s2_smear,_s2_smearing.json,/home/puehlengt/appletree/appletree/maps/_s2_s...,[alt_s2_area_naive],S2 reconstruction smearing


In [24]:
# somehow it's still sr0's g4 inputs..

In [ ]:
aa['option']

In [25]:
# For reference, this is the compiled code, the function is stored in appletree.share._cached_functions
# Initialize component
print('NR')
print(nr.code)

NR
from functools import partial
from jax import jit
from appletree.plugins import BootstrapMS
from appletree.plugins import ThomasImelBox
from appletree.plugins import QyNR
from appletree.plugins import TotalQuanta
from appletree.plugins import LyNR
from appletree.plugins import MeanNphNe
from appletree.plugins import MeanExcitonIon
from appletree.plugins import TrueExcitonIonNR
from appletree.plugins import OmegaNR
from appletree.plugins import MSDriftLoss
from appletree.plugins import TruePhotonElectronNR
from appletree.plugins import MSElectronDrifted
from appletree.plugins import MSElectronMerge
from appletree.plugins import MSPositionRecon
from appletree.plugins import MSS2LCEAlt
from appletree.plugins import MSS2PEAlt
from appletree.plugins import MSS2CorrectionAlt
from appletree.plugins import MSS2Alt
from appletree.plugins import MSS2LCEMain
from appletree.plugins import MSS2PEMain
from appletree.plugins import MSS2CorrectionMain
from appletree.plugins import MSS2Main
from app

In [26]:
nr.compile()

/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load g4_ambe_sr0.npy successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/g4_ambe_sr0.npy
  warn(f"Load {fname} successfully from {fpath}")


There are 319079 events, 1000476 main/alt S2, which are not equal to batch_size.


/home/puehlengt/applefiles/aptext/multiscatter/g4.py:105: UserWarning: Efficiency is not provided in /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/g4_ambe_sr0.npy, set to 1.
  warn(f"Efficiency is not provided in {self.file_path}, set to 1.")


AmBe_llh's map s2_cut_acc is using the parameter s2_cut_acc_sigma.
AmBe_llh's map s1_cut_acc is using the parameter s1_cut_acc_sigma.
AmBe_llh's map s1_eff_n_hits_3f is using the parameter s1_eff_n_hits_3f_sigma.


/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_median_sr0.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_recon_eff_n_hits_median_sr0.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_median_sr0.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_recon_eff_n_hits_median_sr0.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_lower_sr0.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/3fold_recon_eff_n_hits_lower_sr0.json
  warn(f"Load {fname} successfully from {fpath}")
/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load 3fold_recon_eff_n_hits_lower_sr0.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../ap

In [27]:
nr.needed_parameters

{'A',
 'alpha',
 'alpha2',
 'beta',
 'delta',
 'drift_velocity',
 'elife_sigma',
 'epsilon',
 'eta',
 'fano_nex',
 'fano_ni',
 'g1',
 'g2',
 'gamma',
 'gas_gain',
 'hits_acceptance',
 'iota',
 'liquid_xe_density',
 'nr_rate',
 'omega',
 'p_dpe',
 's1_cut_acc_sigma',
 's1_eff_n_hits_3f_sigma',
 's2_cut_acc_sigma',
 's2_threshold',
 'theta',
 'xi',
 'zeta'}

In [30]:
# Of course we have to load parameters (and their priors) in simulation (who the hell writes such comments..)
par_manager = apt.Parameter(get_file_path(instruct['par_config']))

parameters = par_manager.get_all_parameter()

/home/puehlengt/appletree/appletree/utils.py:185: UserWarning: Load param_nr_sr1.json successfully from /home/puehlengt/private_nt_aux_files/ntauxfiles/../apt_files/param_nr_sr1.json
  warn(f"Load {fname} successfully from {fpath}")


In [31]:
parameters

{'gas_gain': 29.4,
 'drift_velocity': 0.0677,
 's2_threshold': 250.0,
 'g1': np.float64(0.13662100011873812),
 'g2': np.float64(17.22069303309113),
 'p_dpe': np.float64(0.19697167527111273),
 'elife_sigma': np.float64(-0.02431607390691891),
 's1_eff_n_hits_3f_sigma': np.float64(0.0057162265135527884),
 's1_cut_acc_sigma': np.float64(1.3190273434579103),
 's2_cut_acc_sigma': np.float64(-1.316677278866778),
 'hits_acceptance': np.float64(0.9310603752973665),
 'liquid_xe_density': 2.8619,
 'alpha': np.float64(10.709088756786112),
 'beta': np.float64(1.0856626582857818),
 'gamma': np.float64(0.04493659282379958),
 'delta': -0.0533,
 'epsilon': np.float64(12.862316908796991),
 'zeta': np.float64(0.3824400986221165),
 'eta': np.float64(2.5645838361707387),
 'theta': np.float64(0.28125819982718525),
 'iota': np.float64(1.0366468019082307),
 'fano_ni': 0.4,
 'fano_nex': 0.4,
 'A': 0.04,
 'xi': 0.5,
 'omega': 0.19,
 'alpha2': 2.25,
 'ambe_nr_rate': np.float64(5510.722681190336)}

In [ ]:
batch_size = int(1e3)
key = apt.randgen.get_key()
key, (cs1, cs2, eff) = nr.simulate(key, batch_size, parameters) # my_param is just a dictionary

2026-03-13 16:42:44.706095: E external/xla/xla/service/slow_operation_alarm.cc:73] Constant folding an instruction is taking > 1s:

  %subtract.786 = f32[1000476,2]{1,0} subtract(f32[1000476,2]{0,1} %constant.9988, f32[1000476,2]{1,0} %broadcast.11773), metadata={op_name="jit(simulate)/jit(main)/jit(simulate)/jit(map_interpolator_regular_binning_2d)/sub" source_file="/home/puehlengt/appletree/appletree/interpolation.py" source_line=138}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
2026-03-13 16:42:46.216157: E external/xla/xla/service/slow_operation_alarm.cc:140] The operation took 2.510431409s
Constant folding an instruction is taking 

In [ ]:
key2 = apt.randgen.get_key(1)
key2, (cs1_2, cs2_2, eff_2) = nr.simulate(key2, batch_size, parameters) # my_param is just a dictionary

In [ ]:
eff.sum(), eff.shape

In [ ]:
33531.88/1000476

In [ ]:
plt.hist2d(cs1, cs2, weights=eff, bins=[np.linspace(0.1, 150, 100), np.geomspace(10**2, 10**4, 100)], norm=LogNorm())
plt.yscale('log')

## Simulation

In [ ]:
num_sigmas = 6
tail_prob = sps.norm.sf(x=num_sigmas)
suggested_max_batch = sps.norm.ppf(1-tail_prob,
                                   loc=par_manager.par_config['nr_rate']['init_mean'],
                                   scale=par_manager.par_config['nr_rate']['init_mean'])
print(suggested_max_batch) # number of NR events hardly gonna fluctuate above this

In [ ]:
batch_size = int(14e3)
#num_sims = int(3000)
num_sims = int(30)

param_bag = []
events_bag = []

for _mc in range(num_sims):
    #print(_mc)
    key = apt.randgen.get_key(seed=_mc)
    par_manager.sample_prior() # sampling from prior
    parameters = par_manager.get_all_parameter()

    key, tmp = nr.simulate(key, batch_size, parameters)
    events = np.asarray(jnp.stack(tmp, axis=1))   # shape (n, 3)
    #print(events)
    param_bag.append(parameters.copy())
    events_bag.append(events)

In [ ]:
target = 200_000

24.6/3000*target/60. # mins

In [ ]:
_mc = 2

this_params = param_bag[_mc]
this_events = events_bag[_mc][:, :2]
this_eff = events_bag[_mc][:, -1].astype(bool)

In [ ]:
this_params, len(this_params)

In [ ]:
plt.plot(this_events[:, 0], this_events[:, 1], '.', label='all events')
plt.plot(this_events[:, 0][~this_eff], this_events[:, 1][~this_eff], 'rx', label='eff = 0')

plt.yscale('log')
plt.xlabel('cs1 [PE]')
plt.ylabel('cs2 [PE]')
plt.legend()

In [ ]:
# todo:
# 0. realistic parameters file
# 1. save the simulated data
# 2. visualize the simulated data
# 3. think of a way to sample number of events according to the rate parameter